In [25]:
#-------------------------------------------------
#   THIS CELL ADDS A JOB TO THE RUNNING JOB LOG
#-------------------------------------------------

from datetime import datetime
import pandas as pd

#-----Collect job information for 1 job order from user. Each of these parameters will repeat for each operation sequence the job contains.
#-----Ops variable does not translate to the job record directly: It informs the number of loop iterations below.

job = int(input("Enter Job Number: "))
part = int(input("Enter Part Number: "))
material = str(input("Aluminum (Enter alm) or Steel? (enter stl)").lower())
due_date = datetime.strptime(input("Enter Due Date (yyyy-mm-dd): "), "%Y-%m-%d")
ops = int(input("How many operations does this job have? "))

#-----Declare an empty list to append dictionaries to. Each dictionary will contain name:value pairs of job parameters for 1 operation sequence.

rows = []

#-----Loop through user input prompts regarding each operation sequence value. Unlike the inputs above, these values will not repeat. Append each value to list 'rows.'
#-----Include the values from the above user inputs. They will be the same in each row generated by the loop.

for x in range(ops):
    y = x + 1
    dept = int(input(f"Which department runs op {y}? "))
    setup_time = float(input("Enter setup time: "))
    run_time = float(input("Enter run time: "))
    row = {"job": job, "part number": part, "material": material, "due date": due_date, "operation sequence": y, "department": dept, "work center": None, "start time": None, "setup time": setup_time, "run time": run_time}
    rows.append(row)

#-----Convert name values from dictionaries in 'rows' to dataframe headers. Fill in dataframe values with the values from each dictionary.

new_rows = pd.DataFrame(rows, columns = ["job", "part number", "material", "due date", "operation sequence", "department", "work center", "start time", "setup time", "run time"])

#-----Append the job parameters stored in dataframe 'new_rows' to running job log --> alljobs.csv

new_rows.to_csv("schedule.csv", mode = 'a', header = False, index = False)

In [26]:
#-----Pull the complete file alljobs.csv into the script. The script now holds all active jobs. If no new job needs to be entered before generating schedule, start here.

import pandas as pd
from datetime import datetime, timedelta
from tabulate import tabulate
from IPython.display import HTML, display
import pandas as pd

def show_df(df):
    display(HTML(tabulate(df, headers='keys', tablefmt='html', stralign='center', numalign='center', showindex=False)))

pd.DataFrame._repr_html_ = lambda self: tabulate(self, headers='keys', tablefmt='html', stralign='center', numalign='center', showindex=False)

schedule = pd.read_csv('schedule.csv', parse_dates = ['due date', 'start time'])                 #pandas df

#-----Load the shop configuration file (shop_config.json) into a dictionary

import json

layout = json.load(open('shop_config.json'))                                                     #python dict

In [27]:
#Building a list of lists. Each list represents an operation sequence in the job entered above. Each index is a work center option.

wc_options = []                                                                    #list of lists

for _, row in new_rows.iterrows():
    candidates = []
    for wc in layout:
        if wc['department'] == row['department'] and row['material'] in wc['capabilities']:
            candidates.append(wc['number'])
    wc_options.append(candidates)

print(wc_options)
show_df(schedule)

[[1, 2], [4, 5], [1, 2]]


job,part number,material,due date,operation sequence,department,work center,start time,setup time,run time
1002,2001,stl,2026-08-19 00:00:00,1,1,1,2026-08-09 07:00:00,1.23,4.56
1002,2001,stl,2026-08-19 00:00:00,2,2,6,2026-08-09 12:40:00,5.12,4.25
1002,2001,stl,2026-08-19 00:00:00,3,1,3,2026-08-09 22:00:00,1.65,5.35
1005,5001,alm,2026-09-09 00:00:00,1,2,4,2026-08-10 07:00:00,1.1,2.34
1005,5001,alm,2026-09-09 00:00:00,2,1,2,2026-08-10 11:00:00,5.44,2.34
1005,5001,alm,2026-09-09 00:00:00,3,2,5,2026-08-10 19:00:00,6.53,9.78
1009,9001,alm,2026-09-15 00:00:00,1,1,nan,NaT,4.56,6.78
1009,9001,alm,2026-09-15 00:00:00,2,2,nan,NaT,1.21,1.23
1009,9001,alm,2026-09-15 00:00:00,3,1,nan,NaT,1.23,8.75


In [28]:
schedule['finish'] = schedule['start time'] + pd.to_timedelta(schedule['setup time'] + schedule['run time'], unit = 'h')

wc_finishes = schedule.groupby('work center')['finish'].max()                          #A series of pairs (wc, latest finish time on that wc)

print(wc_finishes)


work center
1.0   2026-08-09 12:47:24
2.0   2026-08-10 18:46:48
3.0   2026-08-10 05:00:00
4.0   2026-08-10 10:26:24
5.0   2026-08-11 11:18:36
6.0   2026-08-09 22:02:12
Name: finish, dtype: datetime64[ns]


In [29]:
best_wcs = []

for i in wc_options:
    best_wc = int(wc_finishes.loc[i].idxmin())
    best_wcs.append(best_wc)

for x in range(len(best_wcs)):
    op_seq = x + 1
    mask = (schedule['job'] == job) & (schedule['operation sequence'] == op_seq)
    schedule.loc[mask, 'work center'] = best_wcs[x]

print(schedule)

    job  part number material   due date  operation sequence  department  work center          start time  setup time  run time              finish
0  1002         2001      stl 2026-08-19                   1           1          1.0 2026-08-09 07:00:00        1.23      4.56 2026-08-09 12:47:24
1  1002         2001      stl 2026-08-19                   2           2          6.0 2026-08-09 12:40:00        5.12      4.25 2026-08-09 22:02:12
2  1002         2001      stl 2026-08-19                   3           1          3.0 2026-08-09 22:00:00        1.65      5.35 2026-08-10 05:00:00
3  1005         5001      alm 2026-09-09                   1           2          4.0 2026-08-10 07:00:00        1.10      2.34 2026-08-10 10:26:24
4  1005         5001      alm 2026-09-09                   2           1          2.0 2026-08-10 11:00:00        5.44      2.34 2026-08-10 18:46:48
5  1005         5001      alm 2026-09-09                   3           2          5.0 2026-08-10 19:00:00       

In [43]:
idk_what_this_is_doing = []

for x in schedule['work center'].unique():
    matches = schedule[schedule['work center'] == x]

    finish = matches['start time'] + pd.to_timedelta(matches['setup time'] + matches['run time'], unit = 'h')

    def get_end(wc):
        matches = schedule[schedule['work center'] == wc]
        finish = matches['start time'] + pd.to_timedelta(matches['setup time'] + matches['run time'], unit = 'h')
        return finish

    final = wc_options[-1]
    listy = []

    for i in final:
        time = (get_end(i))
        pair = (i, time.max())
        listy.append(pair)

    timeys = []
    for i in listy:
        timeys.append(i[1])

    better_time = min(timeys)

    for i in listy:
        if i[1] == better_time:
            wc_of_champions = i[0]
            idk_what_this_is_doing.append(wc_of_champions)

print(idk_what_this_is_doing)

[1, 1, 1, 1]
